In [ ]:
import time
from openai import OpenAI

# Connect to Microsoft Foundry
client = OpenAI(
    base_url="YOUR OPENAI ENDPOINT",
    api_key="YOUR API KEY"
)

pricing = {
    SMALL_MODEL: {"input": 0.40, "output": 1.60},
    LARGE_MODEL: {"input": 2.00, "output": 8.00}
}

SMALL_MODEL = "gpt-4.1-mini"
LARGE_MODEL = "gpt-4.1"

In [ ]:

# ==========================================
# 1. MODEL CASCADING
# ==========================================

def analyze_complexity(question):
    indicators = [
        len(question.split()) > 50,
        question.count("?") > 1,
        any(word in question.lower() for word in
            ["analyze", "compare", "evaluate", "reasoning"]),
        any(word in question.lower() for word in
            ["technical", "complex", "detailed"])
    ]

    return sum(indicators) / len(indicators)


def choose_model(question):
    score = analyze_complexity(question)

    if score >= 0.5:
        model = LARGE_MODEL
    else:
        model = SMALL_MODEL

    return model, score


def ask_model(question, model=None):
    selected_model, score = choose_model(question)

    if model is None:
        model = selected_model

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": question}
        ],
        max_completion_tokens=250
    )

    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens

    cost = (
        input_tokens * pricing[model]["input"] +
        output_tokens * pricing[model]["output"]
    ) / 1_000_000

    print(f"\nQuestion: {question}")
    print(f"Complexity Score: {score:.2f}")
    print(f"Model: {model}")
    print(f"Input Tokens: {input_tokens}")
    print(f"Output Tokens: {output_tokens}")
    print(f"Estimated Cost: ${cost:.6f}")
    print("Answer:", response.choices[0].message.content)

    return cost


simple_question = "What is the capital of Australia?"

complex_question = (
    "Analyze and compare the technical advantages "
    "of cloud computing and on-premises infrastructure? "
    "What are the detailed differences in security, "
    "scalability, and operational costs?"
)

print("\n========== MODEL CASCADING ==========")

print("\n--- Simple Question: Small Model ---")
cost_small = ask_model(simple_question)

print("\n--- Same Question: Large Model ---")
cost_large = ask_model(simple_question, LARGE_MODEL)

print("\n--- Complex Question: Large Model ---")
cost_complex = ask_model(complex_question)

savings = (1 - cost_small / cost_large) * 100

print("\n--- COST COMPARISON ---")
print(f"Small Model: ${cost_small:.6f}")
print(f"Large Model: ${cost_large:.6f}")
print(f"Cost Savings: {savings:.1f}%")


========== MODEL CASCADING ==========

--- Simple Question: Small Model ---

Question: What is the capital of Australia?
Complexity Score: 0.00
Model: gpt-4.1-mini
Input Tokens: 14
Output Tokens: 7
Estimated Cost: $0.000017
Answer: The capital of Australia is Canberra.

--- Same Question: Large Model ---

Question: What is the capital of Australia?
Complexity Score: 0.00
Model: gpt-4.1
Input Tokens: 14
Output Tokens: 10
Estimated Cost: $0.000108
Answer: The capital of Australia is **Canberra**.

--- Complex Question: Large Model ---

Question: Analyze and compare the technical advantages of cloud computing and on-premises infrastructure? What are the detailed differences in security, scalability, and operational costs?
Complexity Score: 0.75
Model: gpt-4.1
Input Tokens: 36
Output Tokens: 250
Estimated Cost: $0.002072
Answer: Certainly! Here’s a detailed analysis and comparison of **cloud computing** vs **on-premises infrastructure**, focusing on **security**, **scalability**, and **op

In [ ]:
# ==========================================
# 2. PROMPT COMPRESSION
# ==========================================

def compress_context(context):
    response = client.chat.completions.create(
        model=SMALL_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Compress the text while preserving "
                    "all important facts, dates, numbers, "
                    "and conditions."
                )
            },
            {
                "role": "user",
                "content": context
            }
        ],
        max_completion_tokens=300
    )

    print("Compression Input Tokens:",
          response.usage.prompt_tokens)
    print("Compression Output Tokens:",
          response.usage.completion_tokens)

    return response.choices[0].message.content


long_context = """
Customers can return most products within 30 days
of the original purchase date. A valid receipt
or proof of purchase is required for all returns.

Electronics must be unopened, unused, and returned
in their original packaging. Opened electronics
are not eligible for a change-of-mind refund.

Refunds are issued to the original payment method
after the returned item has been inspected and
approved by the returns department.

Damaged or defective products must be reported
within 48 hours of delivery. Customers must
provide photographs of the damage and their
order number.

Shipping fees are generally non-refundable,
except when the product is defective or the
wrong item has been delivered.

Products purchased during promotional sales
follow the same return policy unless otherwise
stated at the time of purchase.
"""

print("\n========== PROMPT COMPRESSION ==========")

compressed_context = compress_context(long_context)

print("\nOriginal Context:")
print(long_context)

print("\nCompressed Context:")
print(compressed_context)

print("\nOriginal Characters:", len(long_context))
print("Compressed Characters:", len(compressed_context))


question = "What is the return policy for electronics?"

print("\n--- Original Context ---")
cost_original = ask_model(
    long_context + "\n\n" + question,
    SMALL_MODEL
)

print("\n--- Compressed Context ---")
cost_compressed = ask_model(
    compressed_context + "\n\n" + question,
    SMALL_MODEL
)

savings = (1 - cost_compressed / cost_original) * 100

print("\n--- PROMPT COMPRESSION COMPARISON ---")
print(f"Original Cost: ${cost_original:.6f}")
print(f"Compressed Cost: ${cost_compressed:.6f}")
print(f"Downstream Savings: {savings:.1f}%")


========== PROMPT COMPRESSION ==========
Compression Input Tokens: 188
Compression Output Tokens: 94

Original Context:

Customers can return most products within 30 days
of the original purchase date. A valid receipt
or proof of purchase is required for all returns.

Electronics must be unopened, unused, and returned
in their original packaging. Opened electronics
are not eligible for a change-of-mind refund.

Refunds are issued to the original payment method
after the returned item has been inspected and
approved by the returns department.

Damaged or defective products must be reported
within 48 hours of delivery. Customers must
provide photographs of the damage and their
order number.

Shipping fees are generally non-refundable,
except when the product is defective or the
wrong item has been delivered.

Products purchased during promotional sales
follow the same return policy unless otherwise
stated at the time of purchase.


Compressed Context:
Customers can return most products 

In [ ]:

# ==========================================
# 3. RESPONSE CACHING
# ==========================================

cache = {}

def get_answer(question):
    if question in cache:
        print("Cache Hit! No API call.")
        print("Additional API Cost: $0.000000")
        return cache[question]

    print("Cache Miss! Calling API...")

    start = time.time()

    response = client.chat.completions.create(
        model=SMALL_MODEL,
        messages=[
            {"role": "user", "content": question}
        ],
        max_completion_tokens=150
    )

    elapsed = time.time() - start

    answer = response.choices[0].message.content
    cache[question] = answer

    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens

    cost = (
        input_tokens * pricing[SMALL_MODEL]["input"] +
        output_tokens * pricing[SMALL_MODEL]["output"]
    ) / 1_000_000

    print(f"Response Time: {elapsed:.2f} seconds")
    print(f"Tokens Used: {response.usage.total_tokens}")
    print(f"Estimated API Cost: ${cost:.6f}")

    return answer


print("\n========== RESPONSE CACHING ==========")

question = "What is the capital of Australia?"

print("\nFirst Request:")
print(get_answer(question))

print("\nSecond Request:")
print(get_answer(question))



========== RESPONSE CACHING ==========

First Request:
Cache Miss! Calling API...
Response Time: 1.88 seconds
Tokens Used: 21
Estimated API Cost: $0.000017
The capital of Australia is Canberra.

Second Request:
Cache Hit! No API call.
Additional API Cost: $0.000000
The capital of Australia is Canberra.
